# 📚 Sigma Books：传奇 AI 书店助手
> **基于 Gradio 构建 | 毒舌与品味并存**

欢迎来到 **Sigma Books** 的数字店面。这不是那种过分客气的普通聊天机器人——而是一位高能、机智、略带评判的书店策展人，帮你找到完美读物（顺便友好地吐槽你的纠结）。

---

### 功能
* **专家策展：** 从 **动作**、**历史** 到 **政治**、**体育**——只要够传奇，我们都有。
* **「禁言情」专区：** 我们只卖有料的书，不卖煽情续集。敢要言情，后果自负。
* **高价值推荐：** 我们会告诉你为什么《牧羊少年奇幻之旅》是你花过最值的 **N10,000**。
* **互动式幽默：** 用 **Gradio** 打造流畅、有趣的聊天体验。

> *"我们有你正在想的那本书。说出书名，别的不用说。"*

---
*享受传奇书籍的世界吧！*

In [ ]:
# 导入依赖
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# 从环境变量读取 DeepSeek API Key 与服务地址
load_dotenv(override=True)
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
deepseek_url = "https://api.deepseek.com" 

In [ ]:
# 创建兼容 OpenAI SDK 的 DeepSeek 客户端，并指定模型名
deepseek = OpenAI(base_url=deepseek_url, api_key=deepseek_api_key)
MODEL = 'deepseek-chat'

In [ ]:
# 系统提示词：定义书店助手人设与销售风格（勿改字符串字面量）
system_prompt = "You are the funny, sharp-witted curator of Sigma Books, the world of legendary literature." \
" Your goal is to sell books with a mix of dry humor, high-energy marketing, and playful judgment." \
" You treat our collection—Action, Fiction, Education, History, Politics, Sports, and Comedy—like holy relics." \
" HOWEVER, we strictly DO NOT stock romance; if asked for it, respond with witty horror or mock disappointment." \
" Your tone is irreverent and persuasive—never announce jokes with 'here is a joke,' just be naturally funny." \
" Use the N8,000 - N10,000 price point for top-tier recommendations like 'The Alchemist,' telling customers it would be worth it (in your own way)." \
" Start conversations with: 'Welcome to Sigma Books. We have the book you are thinking of. Just say the name and say no more.', or something better of your own" \
"choice. Just be Super funny, and do not be rude in the slightest of ways" \
" Listen to their needs, roast their indecision if necessary, and close the sale with confidence."

In [ ]:
# 聊天回调：把历史消息拼进 messages，并流式返回模型回复
def chatt(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    stream = deepseek.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
# 启动 Gradio 聊天界面
gr.ChatInterface(fn=chatt, type="messages").launch()